# 03 - Model TrainingTraining and comparing three regressors (plus XGBoost when installed) toforecast active power.**Chronological splitting.** Time-series data is never shuffled: training onsamples that occur after the test samples lets the model see the future, andthe resulting score is meaningless.    2025-01 .. 2025-08  ->  training    2025-09             ->  testing (held out)

In [ ]:
import sysfrom pathlib import Path# Make the project importable when the notebook runs from notebooks/ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from src.data.preprocess import load_processed_datafrom src.features.feature_engineering import build_training_framefrom src.models.train import build_model_zoo, chronological_splitfrom src.models.evaluate import evaluate_predictions, format_comparison_table, select_best_modelfrom src.utils import configdf = load_processed_data()X, y, names, ts = build_training_frame(df)X_train, X_test, y_train, y_test, ts_train, ts_test = chronological_split(X, y, ts)print(f"Train: {len(X_train):,} samples  {ts_train.min()} -> {ts_train.max()}")print(f"Test : {len(X_test):,} samples  {ts_test.min()} -> {ts_test.max()}")print(f"Overlap: {(ts_train.max() >= ts_test.min())}  (must be False)")

## 1. Why not a random split?The cell below quantifies the damage: a shuffled split inflates the scorebecause near-identical neighbouring hours end up on both sides of the split.

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.ensemble import RandomForestRegressorfrom sklearn.metrics import r2_scoreXr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.2, random_state=42)shuffled = RandomForestRegressor(n_estimators=80, random_state=42, n_jobs=-1).fit(Xr_tr, yr_tr)proper = RandomForestRegressor(n_estimators=80, random_state=42, n_jobs=-1).fit(X_train, y_train)print(f"Random (wrong) split   R2 = {r2_score(yr_te, shuffled.predict(Xr_te)):.4f}")print(f"Chronological split    R2 = {r2_score(y_test, proper.predict(X_test)):.4f}")

## 2. Train every candidate

In [ ]:
results, fitted = {}, {}for name, model in build_model_zoo().items():    model.fit(X_train, y_train)    results[name] = evaluate_predictions(y_test, model.predict(X_test))    fitted[name] = model    print(f"{name:<20} MAE={results[name]['mae']:8.2f}  "          f"RMSE={results[name]['rmse']:8.2f}  R2={results[name]['r2']:.4f}")

In [ ]:
print(format_comparison_table(results))best = select_best_model(results, metric=config.MODEL_SELECTION_METRIC)print(f"\nSelected: {best}  (criterion: lowest test {config.MODEL_SELECTION_METRIC.upper()})")

## 3. Comparison chart

In [ ]:
comparison = pd.DataFrame(results).Tcomparison[["mae", "rmse"]].plot(kind="bar", figsize=(9, 4),                                 title="Model error on the held-out month (lower is better)")plt.ylabel("Watts"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 4. Model selection criterionRMSE is the criterion because it squares the errors before averaging, so amodel that is usually close but occasionally very wrong is penalised. Forpeak-load planning, the occasional large miss is the expensive one.The winner is saved by `python -m src.models.train` as:- `models/best_model.pkl` — the estimator plus its feature list- `models/model_metadata.json` — metrics, split periods, feature names- `models/model_comparison.csv` — the full comparison table

In [ ]:
from src.models.train import train_modelsresults, best_name, metadata = train_models(df)      # writes the artefactsprint(f"Saved '{best_name}' with {metadata['n_features']} features")metadata["metrics"]